# 항만: Kanana-2 vs Qwen 한글 음역 비교 (opus, 10/30)

- 목적: 학습 증강용 한글 음역을 Kanana-2가 Qwen2.5-7B(v1)보다 더 실제 표기에 가깝게 만드는지 확인.
- 프롬프트·few-shot 30개·후처리는 v1(`llm_korean_aug.ipynb`)과 **완전히 동일**. 모델만 바꿈.
- 1단계(약 5분): train 실제 한글 382척 중 few-shot 제외 352척만 생성 → 점수 출력. Qwen v1 기준: exact .290 / any3 .347 / CER .473
- 2단계: 1단계가 확실히 좋을 때만 5,851척 전체 생성 → `llm_korean_aug_kanana.csv` 를 Drive에 저장.

**런타임 → 런타임 유형 변경 → GPU(L4 또는 A100)** 후 위에서부터 실행하세요. 필요한 파일은 기존 Drive 폴더 `항만_llm_aug` 에 이미 있습니다 (`llm_aug_input.csv`, `llm_fewshot.csv`, `train_korean_ref.csv`, `llm_korean_aug.csv`).

In [ ]:
import torch, subprocess
print(torch.cuda.get_device_name())
print(subprocess.run('pip -q install -U transformers accelerate', shell=True, capture_output=True, text=True).stdout[-300:])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/항만_llm_aug'
print(os.listdir(BASE))

In [ ]:
import re, time
import pandas as pd
from difflib import SequenceMatcher

MODEL = 'kakaocorp/kanana-2-3b-instruct'   # 1.3B를 보려면 'kakaocorp/kanana-2-1.3b-instruct'
inp = pd.read_csv(f'{BASE}/llm_aug_input.csv')
fs = pd.read_csv(f'{BASE}/llm_fewshot.csv')
ref = pd.read_csv(f'{BASE}/train_korean_ref.csv')
ref = ref[~ref.vessel_id.isin(fs.vessel_id)].merge(inp, on='vessel_id')
examples = '\n'.join(f'English: {e}\nKorean: {k}' for e, k in zip(fs.en_name, fs.ko_name))
SYSTEM = ('You write the Hangul name that a Korean port authority would record for a foreign or Korean ship. '
          'Transliterate by pronunciation (외래어 표기); do not translate meanings. '
          'Letters of an acronym are read as Korean letter names (MS -> 엠에스, KMTC -> 케이엠티씨). '
          '"No. N" becomes 제N. Answer with the Hangul name only.\n\nExamples:\n' + examples)

def clean(s):
    s = s.strip().split('\n')[0]
    s = re.sub(r'^(Korean|한글)\s*[:：]\s*', '', s)
    s = re.sub(r'[^0-9A-Za-z가-힣 ]', ' ', s)
    return ' '.join(s.split())

from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True); tok.padding_side = 'left'
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map='cuda', trust_remote_code=True).eval()

def prompt(e):
    msgs = [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': f'English: {e}\nKorean:'}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:   # chat template without a system role
        return tok.apply_chat_template([{'role': 'user', 'content': SYSTEM + f'\n\nEnglish: {e}\nKorean:'}],
                                       tokenize=False, add_generation_prompt=True)
print(prompt('MS Freesia')[-400:])

def generate(names, sample, seed, bs=64):
    torch.manual_seed(seed); out = []; t = time.time()
    ps = [prompt(e) for e in names]
    for i in range(0, len(ps), bs):
        enc = tok(ps[i:i + bs], return_tensors='pt', padding=True, add_special_tokens=False).to('cuda')
        with torch.no_grad():
            g = model.generate(**enc, max_new_tokens=24, do_sample=sample, temperature=0.8 if sample else None,
                               top_p=0.95 if sample else None, pad_token_id=tok.pad_token_id)
        out += [clean(x) for x in tok.batch_decode(g[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)]
        if i % (bs * 20) == 0: print(i, len(ps), f'{time.time() - t:.0f}s', flush=True)
    return out

def run_all(df):
    df = df.copy()
    df['ko_greedy'] = generate(df.en_name.tolist(), False, 0)
    df['ko_sample1'] = generate(df.en_name.tolist(), True, 1234)
    df['ko_sample2'] = generate(df.en_name.tolist(), True, 5678)
    return df

## 1단계: 352척 비교 (Qwen v1과 같은 기준)

In [ ]:
nos = lambda s: str(s if isinstance(s, str) else '').replace(' ', '')
def lev(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1): cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]
def score(m, name):
    real = m.ko_name.map(nos); gr = m.ko_greedy.map(nos)
    any3 = pd.concat([m[c].map(nos) == real for c in ('ko_greedy', 'ko_sample1', 'ko_sample2')], axis=1).any(axis=1)
    print(f'{name:10s} n={len(m)} exact {(gr == real).mean():.3f}  any3 {any3.mean():.3f}  '
          f'sim {pd.Series([SequenceMatcher(None, a, b).ratio() for a, b in zip(real, gr)]).mean():.3f}  '
          f'cer {pd.Series([lev(a, b) / max(1, len(a)) for a, b in zip(real, gr)]).mean():.3f}')

kan = run_all(ref[['vessel_id', 'en_name', 'ko_name']])
qwen = ref.merge(pd.read_csv(f'{BASE}/llm_korean_aug.csv', encoding='utf-8-sig').drop(columns='en_name'), on='vessel_id')
score(qwen, 'qwen_v1'); score(kan, 'kanana')
kan.to_csv(f'{BASE}/kanana_eval352.csv', index=False, encoding='utf-8-sig')
d = kan[['en_name', 'ko_name', 'ko_greedy']].assign(qwen=qwen.ko_greedy.values)
print(d.sample(25, random_state=0).to_string())

## 2단계: 1단계에서 Kanana가 확실히 좋을 때만 실행 (5,851척 전체)

In [ ]:
full = run_all(inp)
full.to_csv(f'{BASE}/llm_korean_aug_kanana.csv', index=False, encoding='utf-8-sig')
print(len(full)); print(full.sample(15, random_state=0).to_string())